# 🎯 Pokémon Kaggle Competition: 3-Branch Multi-Modal Neural Network
Welcome! In this notebook, we will build a **Multi-Modal Neural Network** using PyTorch. 
Our goal is to predict if a Pokémon is **Legendary** (1 = Yes, 0 = No) by combining 3 types of data:
1. **Numerical Stats:** (HP, Attack, Defense, etc.)
2. **Image Data (CV):** (The official sprite of the Pokémon)
3. **Text Data (NLP):** (The Pokédex description/flavor text)

In [ ]:
import pandas as pd
import numpy as np
import os
import re
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
import glob

# --- Setup Paths (Kaggle vs Local) ---
if os.path.exists('/kaggle/input'):
    print("Kaggle Environment Detected!")
    # Find the exact directory containing train.csv
    train_path = glob.glob('/kaggle/input/**/train.csv', recursive=True)[0]
    BASE_PATH = os.path.dirname(train_path)
    # Handle the nested images folder Kaggle creates from zip
    IMG_PATH = f'{BASE_PATH}/images/images' if os.path.exists(f'{BASE_PATH}/images/images') else f'{BASE_PATH}/images'
else:
    print("Local Environment Detected!")
    BASE_PATH = 'data'
    IMG_PATH = 'data/images'


## 1. Prepare Text Tokenizer (Vocabulary)
Before we can feed text into a Neural Network, we must convert words into numbers.

In [ ]:
# A simple vocabulary builder
class SimpleTokenizer:
    def __init__(self, max_vocab_size=5000, max_length=50):
        self.max_vocab_size = max_vocab_size
        self.max_length = max_length
        self.word2idx = {'<PAD>': 0, '<UNK>': 1}
        
    def fit(self, texts):
        word_counts = {}
        for text in texts:
            words = self._clean_text(text)
            for w in words:
                word_counts[w] = word_counts.get(w, 0) + 1
                
        # Sort words by frequency
        sorted_words = sorted(word_counts.keys(), key=lambda k: word_counts[k], reverse=True)
        
        # Add top words to vocabulary
        for w in sorted_words[:self.max_vocab_size - 2]:
            self.word2idx[w] = len(self.word2idx)
            
    def _clean_text(self, text):
        text = str(text).lower()
        text = re.sub(r'[^a-z0-9 ]', '', text) # Keep only alphanumeric
        return text.split()
        
    def encode(self, text):
        words = self._clean_text(text)
        # Convert words to indices, use <UNK> if word not in vocab
        indices = [self.word2idx.get(w, self.word2idx['<UNK>']) for w in words]
        
        # Truncate or Pad
        if len(indices) > self.max_length:
            indices = indices[:self.max_length]
        else:
            indices = indices + [0] * (self.max_length - len(indices))
            
        return indices

# Load training text to build vocabulary
train_df_temp = pd.read_csv(f'{BASE_PATH}/train.csv')
tokenizer = SimpleTokenizer(max_length=40)
tokenizer.fit(train_df_temp['Description'])
vocab_size = len(tokenizer.word2idx)
print(f"Vocabulary Size: {vocab_size}")

## 2. Prepare Dataset Class
We create a `Dataset` class that handles Images, Numerical Stats, and Tokenized Text.

In [ ]:
num_features = ['HP', 'Attack', 'Defense', 'Sp_Atk', 'Sp_Def', 'Speed', 'Weight', 'Height']

class PokemonMultiModalDataset(Dataset):
    def __init__(self, csv_file, img_dir, tokenizer, scaler=None, is_train=True):
        self.df = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.tokenizer = tokenizer
        self.is_train = is_train
        
        self.scaler = scaler
        if self.scaler is None:
            self.scaler = StandardScaler()
            self.scaled_num = self.scaler.fit_transform(self.df[num_features])
        else:
            self.scaled_num = self.scaler.transform(self.df[num_features])
            
        self.transform = transforms.Compose([
            transforms.Resize((64, 64)),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # 1. Load Image
        img_name = os.path.join(self.img_dir, self.df.iloc[idx]['Image_File'])
        image = Image.open(img_name).convert('RGB')
        image = self.transform(image)
        
        # 2. Load Stats
        stats = torch.tensor(self.scaled_num[idx], dtype=torch.float32)
        
        # 3. Load & Encode Text
        text = self.df.iloc[idx]['Description']
        text_indices = torch.tensor(self.tokenizer.encode(text), dtype=torch.long)
        
        if self.is_train:
            label = torch.tensor(self.df.iloc[idx]['Is_Legendary'], dtype=torch.float32)
            return image, stats, text_indices, label
        else:
            return image, stats, text_indices

# Create DataLoaders
train_dataset = PokemonMultiModalDataset(f'{BASE_PATH}/train.csv', IMG_PATH, tokenizer)
test_dataset = PokemonMultiModalDataset(f'{BASE_PATH}/test.csv', IMG_PATH, tokenizer, scaler=train_dataset.scaler, is_train=False)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
print("DataLoaders Ready!")

## 3. Build the 3-Branch Multi-Modal Network
- **Image Branch (CV):** Convolutional Neural Network (`Conv2d` -> `MaxPool2d` -> `Flatten` -> `Linear`)
- **Stats Branch (Tabular):** `Linear`
- **Text Branch (NLP):** `nn.Embedding` -> Global Average Pooling -> `Linear`
- **Merge:** Concatenate all 3 branches! We use `Dropout` to prevent overfitting.

In [ ]:
class MultiModalNN(nn.Module):
    def __init__(self, num_stats, vocab_size, embed_dim=32):
        super(MultiModalNN, self).__init__()
        
        # 1. Image Branch (CNN) - Input shape: (3, 64, 64)
        self.img_branch = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # Output: (16, 32, 32)
            
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # Output: (32, 16, 16)
            
            nn.Flatten(),
            nn.Linear(32 * 16 * 16, 128),
            nn.ReLU()
        )
        
        # 2. Stats Branch
        self.stats_branch = nn.Sequential(
            nn.Linear(num_stats, 32),
            nn.ReLU()
        )
        
        # 3. Text (NLP) Branch
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.text_branch = nn.Sequential(
            nn.Linear(embed_dim, 32),
            nn.ReLU()
        )
        
        # 4. Merged Classifier
        self.classifier = nn.Sequential(
            nn.Linear(128 + 32 + 32, 64),
            nn.ReLU(),
            nn.Dropout(0.3), # Regularization to prevent overfitting
            nn.Linear(64, 1) # No Sigmoid here because we use BCEWithLogitsLoss
        )
        
    def forward(self, image, stats, text):
        img_out = self.img_branch(image)
        stats_out = self.stats_branch(stats)
        
        # Text processing: Embed words, then take the average over the sequence length
        embedded = self.embedding(text) # shape: (batch_size, seq_len, embed_dim)
        text_avg = embedded.mean(dim=1) # shape: (batch_size, embed_dim) (Global Average Pooling)
        text_out = self.text_branch(text_avg)
        
        # Merge
        merged = torch.cat((img_out, stats_out, text_out), dim=1)
        output = self.classifier(merged)
        return output

model = MultiModalNN(num_stats=len(num_features), vocab_size=vocab_size)

# Handle severe class imbalance (289 Normal vs 11 Legendary = ~26:1 ratio)
pos_weight = torch.tensor([26.0])
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=0.001)
print("Model Ready!")

In [ ]:
# Train the model
epochs = 12 # Hint: You might need to train for more epochs!
model.train()

for epoch in range(epochs):
    epoch_loss = 0
    all_preds = []
    all_labels = []
    
    for images, stats, text_indices, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images, stats, text_indices).squeeze()
        
        # Handle case where batch size = 1 (squeeze might remove batch dim)
        if outputs.dim() == 0: outputs = outputs.unsqueeze(0)
            
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
        # Apply sigmoid to convert logits to probabilities
        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).float()
        
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())
        
    epoch_f1 = f1_score(all_labels, all_preds)
    print(f"Epoch [{epoch+1}/{epochs}] | Loss: {epoch_loss/len(train_loader):.4f} | F1-Score: {epoch_f1:.4f}")

## 4. Generate Submission File

In [ ]:
model.eval()
test_preds = []

with torch.no_grad():
    for images, stats, text_indices in test_loader:
        outputs = model(images, stats, text_indices).squeeze()
        if outputs.dim() == 0: outputs = outputs.unsqueeze(0)
        
        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).int().numpy()
        test_preds.extend(preds)

submission = pd.read_csv(f'{BASE_PATH}/sample_submission.csv')
submission['Is_Legendary'] = test_preds
submission.to_csv('submission.csv', index=False)
print("submission.csv successfully created!")